# Chapter 1: Static Scraping Basics

This notebook covers the fundamentals of static web scraping using `httpx` and `BeautifulSoup`.
We work against the **ShopSphere SSR** demo site (server-side rendered) at `localhost:8001`.

**Prerequisites:** Start the demo sites first.
```bash
cd demo-sites && docker compose up -d
```

**What you will learn:**
- How an HTTP request retrieves HTML from a server
- How `BeautifulSoup` parses that HTML into a navigable tree
- How to extract text, attributes, and structured data
- Why SSR pages are straightforward to scrape

In [1]:
import httpx
from bs4 import BeautifulSoup
import json

BASE_URL = "http://localhost:8001"

# Verify the site is up
resp = httpx.get(f"{BASE_URL}/")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}")
print(f"Response size: {len(resp.text):,} bytes")

Status: 200
Content-Type: text/html; charset=utf-8
Response size: 17,217 bytes


## 1.1 The Raw HTML Response

On an SSR site, the server sends back a complete HTML document. All the product names, prices, and links are in the HTML string - no JavaScript execution needed.

In [2]:
html = resp.text

# SSR pages contain real content in the raw HTML
print("Contains 'product-card':", 'product-card' in html)
print("Contains 'MacBook':", 'MacBook' in html)
print("Contains '$1,999':", '1999' in html)
print()
# Show a slice of the HTML to confirm structure is present
start = html.find('class="product-card"')
print(html[start:start+300])

Contains 'product-card': True
Contains 'MacBook': True
Contains '$1,999': True

class="product-card" data-product-id="PROD-001">
        <div class="product-image-wrapper">
          <img class="product-image"
               src="https://picsum.photos/seed/macbook-pro-14/400/200"
               alt="MacBook Pro 14&#34;"
               loading="lazy">
          <span class="prod


## 1.2 Parsing with BeautifulSoup

`BeautifulSoup` turns the raw HTML string into a tree of Python objects you can query with CSS selectors or DOM traversal methods.

In [3]:
soup = BeautifulSoup(html, "lxml")

# The page title
print("Title:", soup.title.string)
print()

# Count product cards on this page
cards = soup.select("article.product-card")
print(f"Product cards on page 1: {len(cards)}")
print()

# Extract product links
links = soup.select("a.product-link")
print(f"Product links: {len(links)}")
for link in links[:3]:
    print(f"  {link.get_text(strip=True)[:50]} -> {link['href']}")

Title: Products - ShopSphere

Product cards on page 1: 10

Product links: 10
  MacBook Pro 14" -> /products/macbook-pro-14
  Dell XPS 15 -> /products/dell-xps-15
  Lenovo ThinkPad X1 Carbon Gen 11 -> /products/lenovo-thinkpad-x1-carbon


## 1.3 Extracting Structured Data from a Listing Page

In [4]:
listing_data = []

for card in cards:
    name = card.select_one("h2.product-name a")
    price = card.select_one(".product-price")
    rating = card.select_one(".product-rating")
    reviews = card.select_one(".product-reviews")
    stock = card.select_one(".product-stock")
    href = name["href"] if name else ""
    
    listing_data.append({
        "name": name.get_text(strip=True) if name else None,
        "url": f"{BASE_URL}{href}" if href.startswith("/") else href,
        "price": price.get_text(strip=True) if price else None,
        "rating": rating.get_text(strip=True) if rating else None,
        "in_stock": "In Stock" in stock.get_text() if stock else None,
    })

for item in listing_data:
    print(item)

{'name': 'MacBook Pro 14"', 'url': 'http://localhost:8001/products/macbook-pro-14', 'price': '$1999.00', 'rating': '4.8/5', 'in_stock': True}
{'name': 'Dell XPS 15', 'url': 'http://localhost:8001/products/dell-xps-15', 'price': '$1499.00', 'rating': '4.6/5', 'in_stock': True}
{'name': 'Lenovo ThinkPad X1 Carbon Gen 11', 'url': 'http://localhost:8001/products/lenovo-thinkpad-x1-carbon', 'price': '$1299.00', 'rating': '4.7/5', 'in_stock': True}
{'name': 'ASUS ZenBook Pro Duo 15', 'url': 'http://localhost:8001/products/asus-zenbook-pro-duo', 'price': '$1799.00', 'rating': '4.4/5', 'in_stock': False}
{'name': 'iPhone 15 Pro', 'url': 'http://localhost:8001/products/iphone-15-pro', 'price': '$999.00', 'rating': '4.9/5', 'in_stock': True}
{'name': 'Samsung Galaxy S24 Ultra', 'url': 'http://localhost:8001/products/samsung-galaxy-s24-ultra', 'price': '$1299.00', 'rating': '4.7/5', 'in_stock': True}
{'name': 'Google Pixel 8 Pro', 'url': 'http://localhost:8001/products/google-pixel-8-pro', 'price

## 1.4 Fetching a Detail Page

The listing page gives us links. Each link leads to a detail page with the full product data.

In [5]:
# Grab the first product URL
detail_url = listing_data[0]["url"]
print(f"Fetching: {detail_url}")

detail_resp = httpx.get(detail_url)
detail_soup = BeautifulSoup(detail_resp.text, "lxml")

# Extract product details
product = {
    "title": detail_soup.select_one("h1.product-title").get_text(strip=True),
    "price": detail_soup.select_one("span.price-amount").get_text(strip=True),
    "rating": detail_soup.select_one("span.rating-value")["data-rating"],
    "review_count": detail_soup.select_one("span.review-count").get_text(strip=True),
    "in_stock": detail_soup.select_one("span.stock-status").get_text(strip=True),
    "description": detail_soup.select_one("p.product-description-full").get_text(strip=True),
    "category": detail_soup.select_one("span.product-category-badge.large").get_text(strip=True),
    "tags": [t.get_text(strip=True) for t in detail_soup.select("span.tag")],
}

# Specs from table
specs = {}
for row in detail_soup.select("table.specs-table tr"):
    key = row.select_one("td.spec-key")
    val = row.select_one("td.spec-value")
    if key and val:
        specs[key.get_text(strip=True)] = val.get_text(strip=True)
product["specs"] = specs

print(json.dumps(product, indent=2))

Fetching: http://localhost:8001/products/macbook-pro-14
{
  "title": "MacBook Pro 14\"",
  "price": "$1999.00",
  "rating": "4.8",
  "review_count": "(2341 customer reviews)",
  "in_stock": "In Stock",
  "description": "The most powerful MacBook Pro ever. M3 Pro chip with 18GB unified memory delivers exceptional performance for pro workloads.",
  "category": "Laptops",
  "tags": [
    "apple",
    "laptop",
    "professional"
  ],
  "specs": {
    "Processor": "Apple M3 Pro",
    "RAM": "18GB",
    "Storage": "512GB SSD",
    "Display": "14.2\" Liquid Retina XDR",
    "Battery": "Up to 18 hours"
  }
}


## 1.5 Crawling All Pages (Pagination)

The listing has 5 pages of 10 products each. We iterate through pages until no more products are found.

In [6]:
all_products = []

for page in range(1, 6):
    url = f"{BASE_URL}/products?page={page}"
    resp = httpx.get(url)
    soup = BeautifulSoup(resp.text, "lxml")
    
    links = soup.select("a.product-link")
    if not links:
        print(f"Page {page}: no links found, stopping")
        break
    
    print(f"Page {page}: {len(links)} products")
    
    for link in links:
        href = link.get("href", "")
        full_url = f"{BASE_URL}{href}" if href.startswith("/") else href
        all_products.append({"name": link.get_text(strip=True), "url": full_url})

print(f"\nTotal products found: {len(all_products)}")

Page 1: 10 products
Page 2: 10 products
Page 3: 10 products
Page 4: 10 products
Page 5: 10 products

Total products found: 50


## 1.6 Key Takeaways

1. **SSR pages are straightforward**: one HTTP request returns all the data you need
2. **BeautifulSoup + CSS selectors** give you a clean query interface over HTML
3. **Pagination** is just repeating the same request with an incrementing page number
4. **Two-pass scraping**: collect URLs from listing pages, then fetch each detail page
5. **The limit**: this only works when the server puts the data in the HTML. Chapter 4 covers what happens when it does not.

In [7]:
# What does a CSR site look like to a static scraper?
csr_resp = httpx.get("http://localhost:8002/products")
csr_soup = BeautifulSoup(csr_resp.text, "lxml")

csr_cards = csr_soup.select(".product-card")
csr_text = csr_soup.get_text(strip=True)

print(f"CSR - product cards found by BeautifulSoup: {len(csr_cards)}")
print(f"CSR - visible text length: {len(csr_text)} chars")
print(f"CSR - visible text: {csr_text[:200]!r}")
print()
print("The CSR site sends only the empty shell. JavaScript renders the content.")
print("BeautifulSoup sees nothing useful. Playwright is the solution - see Notebook 04.")

CSR - product cards found by BeautifulSoup: 0
CSR - visible text length: 159 chars
CSR - visible text: 'ShopSphereShopSphereAll ProductsCSRLoading productsShopSphere CSR Demo - Content rendered client-side via JavaScript. View source to confirm the HTML is empty.'

The CSR site sends only the empty shell. JavaScript renders the content.
BeautifulSoup sees nothing useful. Playwright is the solution - see Notebook 04.
